# Week 01 -- Day 04: Reshaping Data for Visualization

In [1]:
# To list files in a directory
import os 

# For data manipulation
import pandas as pd

import numpy as np

 

Create a new Python to load the data from the ME204/data/waitrose folder into your notebook using the code below:

In [2]:
# List all files in the ME204/data/waitrose folder
all_files = [os.path.join('../data/waitrose_data/waitrose', file) for file in os.listdir('../data/waitrose_data/waitrose') 
             if file.endswith('.csv')]



In [3]:
all_files

['../data/waitrose_data/waitrose\\baby-child-and-parent.csv',
 '../data/waitrose_data/waitrose\\bakery.csv',
 '../data/waitrose_data/waitrose\\beer-wine-and-spirits.csv',
 '../data/waitrose_data/waitrose\\best-of-british.csv',
 '../data/waitrose_data/waitrose\\dietary-and-lifestyle.csv',
 '../data/waitrose_data/waitrose\\everyday-value.csv',
 '../data/waitrose_data/waitrose\\food-cupboard.csv',
 '../data/waitrose_data/waitrose\\fresh-and-chilled.csv',
 '../data/waitrose_data/waitrose\\frozen.csv',
 '../data/waitrose_data/waitrose\\home.csv',
 '../data/waitrose_data/waitrose\\household.csv',
 '../data/waitrose_data/waitrose\\new.csv',
 '../data/waitrose_data/waitrose\\organic-shop.csv',
 '../data/waitrose_data/waitrose\\pet.csv',
 '../data/waitrose_data/waitrose\\summer.csv',
 '../data/waitrose_data/waitrose\\tea-coffee-and-soft-drinks.csv',
 '../data/waitrose_data/waitrose\\toiletries-health-and-beauty.csv',
 '../data/waitrose_data/waitrose\\waitrose-brands.csv']

In [4]:
# Read every single file and concatenate them into a single DataFrame with pandas concat
df = pd.concat((pd.read_csv(file) for file in all_files))
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25418 entries, 0 to 1593
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   data-product-id        25418 non-null  int64  
 1   data-product-name      25418 non-null  object 
 2   data-product-type      25418 non-null  object 
 3   data-product-on-offer  25418 non-null  bool   
 4   data-product-index     25408 non-null  float64
 5   image-url              25418 non-null  object 
 6   product-page           25418 non-null  object 
 7   product-name           25407 non-null  object 
 8   product-size           25363 non-null  object 
 9   item-price             25407 non-null  object 
 10  price-per-unit         24976 non-null  object 
 11  offer-description      7201 non-null   object 
 12  category               25418 non-null  object 
dtypes: bool(1), float64(1), int64(1), object(10)
memory usage: 2.5+ MB


Check that the data was loaded correctly by running the following code:

In [5]:
df.head()

,data-product-id,data-product-name,data-product-type,data-product-on-offer,data-product-index,image-url,product-page,product-name,product-size,item-price,price-per-unit,offer-description,category
0,525635,Organix Raspberry & Apple Soft Oaty Bars,G,False,1.0,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/organix...,Organix Raspberry & Apple Soft Oaty Bars,6x23g,£3.15,£2.29/100g,NaN,"Baby, Child & Parent"
1,557746,Organix Carrot Cake Oaty Bars,G,False,2.0,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/organix...,Organix Carrot Cake Oaty Bars,6x23g,£3.15,£2.29/100g,NaN,"Baby, Child & Parent"
2,32062,Aptamil 2 Follow On Milk,G,False,394.0,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/aptamil...,Aptamil 2 Follow On Milk,800g,£13.50,£16.88/kg,NaN,"Baby, Child & Parent"
3,767801,Essential Baby Wipes,G,False,4.0,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/essenti...,Essential Baby Wipes,80s,95p,1.2p each,NaN,"Baby, Child & Parent"
4,514054,Organix Apple Rice Cakes,G,False,5.0,https://ecom-su-static-prod.wtrecom.com/images...,https://www.waitrose.com/ecom/products/organix...,Organix Apple Rice Cakes,40g,£1.60,£4/100g,NaN,"Baby, Child & Parent"


You should see the first few rows of the dataset. You can also use df.info() to get more information about the dataset.

Perform the initial pre-processing steps to clean the data using the following code

In [6]:
# Drop duplicates
df = df.drop_duplicates()

df = df.drop(columns=['data-product-name', 
                      'data-product-type', 
                      'data-product-index'])
df = (
    df.rename(columns={
        'data-product-id': 'id',
        'data-product-on-offer': 'offer',
        'product-page': 'page',
        'product-name': 'name',
        'product-size': 'size',
    })
)

# The id does not need 64 bits. 32 bits is enough.
# See https://numpy.org/doc/stable/reference/arrays.scalars.html#numpy.intc for ranges.
df['id'] = df['id'].astype('int32')

## 📋 Exercise 01: Revisiting the clean_item_price() function

Yesterday, we created the following function to clean the item price and it worked fine for the bakery.csv file:

In [7]:
def clean_item_price(item_price: str):    
    """
    Cleans the item price string by performing necessary transformations.

    Parameters:
    item_price (str): The item price as a string.

    Returns:
    str: The cleaned item price.
    """
    
    if '£' in item_price:
        item_price = item_price.replace('£', '')
    elif 'p' in item_price:
        item_price = item_price.replace('p', '')
        item_price = '0.' + item_price
        
    return float(item_price)

Now, you will notice that it no longer works when we have more data. We will need to rethink the function to make it more robust.

### 🎯 ACTION POINTS:

In conversation with your coding partner, try to figure out and articulate why the error is happening

In [8]:
df.columns

Index(['id', 'offer', 'image-url', 'page', 'name', 'size', 'item-price',
       'price-per-unit', 'offer-description', 'category'],
      dtype='object')

In [9]:
df[['item-price']]

,item-price
0,£3.15
1,£3.15
2,£13.50
3,95p
4,£1.60
...,...
1589,£24.31 each est.
1590,£2.40
1591,£10.00
1592,£2.50


In [10]:
def clean_item_price(item_price: str):
        
    if 'each est.' in item_price:
        item_price = item_price.replace('each est.', '')


    if type(item_price) != str:
        pass
    elif '£' in item_price:
        item_price = item_price.replace('£', '')
    elif 'p' in item_price:
        item_price = item_price.replace('p', '')
        item_price = '0.'+ item_price

    # Treat cases with hyphen
    if '-' in item_price:
        #TODO: This implies I also must change the 'size' column
        item_price = item_price.split('-')[0]

    return float(item_price)

Spot which rows/columns are causing the issue.

Make decisions about how to handle the issue without dropping any data.

Adjust the clean_item_price() function to handle the new situations you found.

Check that you have a new df['item-price'] column with the cleaned prices.

I will save the cleaned column to a new column item-price-fixed because I will need the original data to cclean up the size column(see TODO above).

In [11]:
# Apply the function to the item price column
df['item-price-fixed'] = df['item-price'].astype(str).apply(clean_item_price)

In [12]:
selected_rows = df['item-price-fixed'].astype(str).str.contains('-') # to check if there is any hyphenated values in item-price-fixed

Time to adjust the size column

In [13]:
selected_rows = df['item-price'].astype(str).str.contains('-')

# The 'size' column of the selected rows all have the same kind of format: (2.2kg-2.8kg).
# I need to get the information that comes before the hyphen (but drop the parentheses).

df.loc[selected_rows,'size'] = df.loc[selected_rows, 'size'].str.split('-').str[0].str.replace('(','')

Remove the original item-price column and rename the item-price-fixed to item-price.

In [14]:
df['item-price'] = df['item-price-fixed']

In [15]:
df['item-price'] 

0        3.15
1        3.15
2       13.50
3        0.95
4        1.60
        ...  
1589    24.31
1590     2.40
1591    10.00
1592     2.50
1593    21.49
Name: item-price, Length: 25378, dtype: float64

In [16]:
df['item-price-fixed'] = df['item-price'].astype(str).apply(clean_item_price) #item-price-fixed is stored as float

In [17]:
df.drop(columns= ['item-price-fixed'],inplace = True)

## 📋 Exercise 02: What is the distribution of prices per category?

We now have more categories in the dataset. We would like to know the distribution of prices per category and eventually plot it.

In [18]:
df.groupby('category')['item-price'].describe()

,count,mean,std,min,25%,50%,75%,max
category,,,,,,,,
"Baby, Child & Parent",428.0,5.196215,4.677661,0.60,1.8000,3.075,7.2625,25.00
Bakery,482.0,4.846680,7.750208,0.50,1.6000,2.200,3.1500,45.00
"Beer, Wine & Spirits",1679.0,14.425057,15.720801,0.30,5.3700,9.990,17.9900,299.99
Best of British,322.0,6.928106,12.570727,0.10,2.7125,4.150,6.0000,168.00
Dietary & Lifestyle,3337.0,3.795999,4.755322,0.10,1.9500,2.750,4.0000,139.99
Everyday Value,141.0,1.740284,1.202873,0.15,1.1000,1.350,2.0000,8.00
Food Cupboard,4186.0,2.665160,2.138691,0.35,1.6000,2.250,3.0000,43.50
Fresh & Chilled,3578.0,3.815151,5.267549,0.10,2.0000,2.900,4.1425,168.00
Frozen,431.0,3.627378,1.544523,0.70,2.6500,3.500,4.4750,12.00


In [19]:
df.groupby('category')['item-price'].describe().reset_index()

,category,count,mean,std,min,25%,50%,75%,max
0,"Baby, Child & Parent",428.0,5.196215,4.677661,0.60,1.8000,3.075,7.2625,25.00
1,Bakery,482.0,4.846680,7.750208,0.50,1.6000,2.200,3.1500,45.00
2,"Beer, Wine & Spirits",1679.0,14.425057,15.720801,0.30,5.3700,9.990,17.9900,299.99
3,Best of British,322.0,6.928106,12.570727,0.10,2.7125,4.150,6.0000,168.00
4,Dietary & Lifestyle,3337.0,3.795999,4.755322,0.10,1.9500,2.750,4.0000,139.99
5,Everyday Value,141.0,1.740284,1.202873,0.15,1.1000,1.350,2.0000,8.00
6,Food Cupboard,4186.0,2.665160,2.138691,0.35,1.6000,2.250,3.0000,43.50
7,Fresh & Chilled,3578.0,3.815151,5.267549,0.10,2.0000,2.900,4.1425,168.00
8,Frozen,431.0,3.627378,1.544523,0.70,2.6500,3.500,4.4750,12.00
9,Home,1075.0,8.595033,10.764164,0.49,3.0000,5.990,10.5000,159.00


In [20]:
plot_df = df.groupby('category')['item-price'].describe().reset_index()

In [21]:
plot_df

,category,count,mean,std,min,25%,50%,75%,max
0,"Baby, Child & Parent",428.0,5.196215,4.677661,0.60,1.8000,3.075,7.2625,25.00
1,Bakery,482.0,4.846680,7.750208,0.50,1.6000,2.200,3.1500,45.00
2,"Beer, Wine & Spirits",1679.0,14.425057,15.720801,0.30,5.3700,9.990,17.9900,299.99
3,Best of British,322.0,6.928106,12.570727,0.10,2.7125,4.150,6.0000,168.00
4,Dietary & Lifestyle,3337.0,3.795999,4.755322,0.10,1.9500,2.750,4.0000,139.99
5,Everyday Value,141.0,1.740284,1.202873,0.15,1.1000,1.350,2.0000,8.00
6,Food Cupboard,4186.0,2.665160,2.138691,0.35,1.6000,2.250,3.0000,43.50
7,Fresh & Chilled,3578.0,3.815151,5.267549,0.10,2.0000,2.900,4.1425,168.00
8,Frozen,431.0,3.627378,1.544523,0.70,2.6500,3.500,4.4750,12.00
9,Home,1075.0,8.595033,10.764164,0.49,3.0000,5.990,10.5000,159.00


In [22]:
plot_df.rename(columns ={
    "25%" : "Q1",
    "50%" : "median",
    "75%" : "Q3"
},inplace = True)

In [23]:
plot_df

,category,count,mean,std,min,Q1,median,Q3,max
0,"Baby, Child & Parent",428.0,5.196215,4.677661,0.60,1.8000,3.075,7.2625,25.00
1,Bakery,482.0,4.846680,7.750208,0.50,1.6000,2.200,3.1500,45.00
2,"Beer, Wine & Spirits",1679.0,14.425057,15.720801,0.30,5.3700,9.990,17.9900,299.99
3,Best of British,322.0,6.928106,12.570727,0.10,2.7125,4.150,6.0000,168.00
4,Dietary & Lifestyle,3337.0,3.795999,4.755322,0.10,1.9500,2.750,4.0000,139.99
5,Everyday Value,141.0,1.740284,1.202873,0.15,1.1000,1.350,2.0000,8.00
6,Food Cupboard,4186.0,2.665160,2.138691,0.35,1.6000,2.250,3.0000,43.50
7,Fresh & Chilled,3578.0,3.815151,5.267549,0.10,2.0000,2.900,4.1425,168.00
8,Frozen,431.0,3.627378,1.544523,0.70,2.6500,3.500,4.4750,12.00
9,Home,1075.0,8.595033,10.764164,0.49,3.0000,5.990,10.5000,159.00


In [24]:
from lets_plot import *
LetsPlot.setup_html()

In [25]:

# This configures what shows up when you hover your mouse over the plot.
tooltip_setup = (
    layer_tooltips()
        .line('@category')
        .line('[@Q1 -- @median -- @Q3]')
        .format('@Q1', '£ {.2f}')
        .format('@median', '£ {.2f}')
        .format('@Q3', '£ {.2f}')
)

g = (
    # Maps the columns to the aesthetics of the plot.
    ggplot(plot_df, aes(y='category', x='median', xmin='Q1', xmax='Q3', fill='category')) +

    # GEOMS

    # Add a line range that 'listens to' columns informed in `ymin` and `ymax` aesthetics
    geom_linerange(size=1, alpha=0.75, tooltips=tooltip_setup) +

    # Add points to the plot (listen to `x` and `y` and fill aesthetics)
    geom_point(size=3, stroke=1, shape=21, tooltips=tooltip_setup) +

    # SCALES

    # Remove the legend (we can already read the categories from the y-axis)
    scale_fill_discrete(guide='none') +

    # Specify names for the axes
    scale_y_continuous(name="Categories\n(from largest to smallest median)", expand=[0.05, 0.05]) +
    scale_x_continuous(name="Price (£)", expand=[0., 0.05], format='£ {.2f}', breaks=np.arange(0, 20, 2.5)) +

    # LABELS
    # It's nice when the plot tells you the key takeaways
    labs(title='"Beer, Wine & Spirits" has the highest median price for individual items',
         subtitle="Dots represent the median price, bars represent the 25th and 75th percentiles") +
    theme(axis_text_x=element_text(size=15),
        axis_text_y=element_text(size=17),
        axis_title_x=element_text(size=20),
        axis_title_y=element_text(size=20),
        plot_title=element_text(size=19, face='bold'),
        plot_subtitle=element_text(size=18),
        legend_position='none') +
    ggsize(1000, 500)

)

g